In [ ]:
import pandas as pd
import numpy as np
import random
from ortools.linear_solver import pywraplp


# 1. LOAD AND PREP THE SYNTHETIC DATA
try:
    employees_df = pd.read_csv('../dataset_exports//2026-03-30_17-39-18/employees.csv')
    tasks_df = pd.read_csv('../dataset_exports/2026-03-30_17-39-18/tasks.csv')
except FileNotFoundError:
    print("Error: Could not find the CSVs. Make sure you ran the generation script first!")
    raise

# Convert the pipe-separated strings back into Python lists
employees_df['Categories'] = employees_df['Categories'].str.split('|')

# OR TRICK: Inject the External Contractor to guarantee feasibility
# We give them every category that exists in the task list
all_categories = list(tasks_df['Category_Needed'].unique())

TAXONOMY = {
    'Backend': ['Python', 'Java', 'Go', 'Node.js', 'C#'],
    'Frontend': ['React', 'Angular', 'Vue.js', 'TypeScript'],
    'Testing': ['PyTest', 'Selenium', 'Cypress', 'JUnit'],
    'DevOps': ['AWS', 'Docker', 'Kubernetes', 'Terraform'],
    'Data': ['SQL', 'Pandas', 'Spark', 'Tableau']
}

# Sposób 1: Czytelny string rozdzielony przecinkami
all_skills_string = ", ".join([skill for sublist in TAXONOMY.values() for skill in sublist])

# Sposób 2: Format pasujący do Twojego kodu (rozdzielony '|')
all_skills_pipe = "|".join([skill for sublist in TAXONOMY.values() for skill in sublist])

# print(all_skills_pipe)
contractor = pd.DataFrame([{
    'Employee_ID': 'EXT_CONTRACTOR',
    'Categories': all_categories,
    'Specific_Skills': all_skills_pipe, 
    'Hourly_Cost': 5000,
    'Max_Hours': 999999
}])
employees_df = pd.concat([employees_df, contractor], ignore_index=True)

# print(employees_df.head(5))
# print(tasks_df.head(5))

  Employee_ID          Categories                Specific_Skills  Hourly_Cost  \
0        E001  [Backend, Testing]                      Go|PyTest           67   
1        E002    [Data, Frontend]              SQL|Tableau|React           86   
2        E003              [Data]                 Pandas|Tableau           65   
3        E004  [Frontend, DevOps]  TypeScript|Angular|Vue.js|AWS           97   
4        E005            [DevOps]                     Kubernetes           58   

   Max_Hours  
0       2080  
1       2080  
2       2080  
3       2080  
4       2080  
  Project_ID  Task_ID Category_Needed  Skills_Needed  Hours_Needed
0       P001  P001_T1         Testing          JUnit           400
1       P001  P001_T2         Backend     C#|Node.js           600
2       P002  P002_T1            Data  Spark|Tableau           800
3       P002  P002_T2            Data    Tableau|SQL           200
4       P003  P003_T1         Backend    Java|Python           300


In [102]:
# 3. Generate Valid Pairs using Set Theory
valid_pairs = []
for _, emp in employees_df.iterrows():
    emp_skill_set = set(emp['Specific_Skills']) 
    for _, task in tasks_df.iterrows():
        task_req_set = set(task['Skills_Needed'])
        # subset check: employee must have ALL skills required by the task
        if task_req_set.issubset(emp_skill_set):
            valid_pairs.append((emp['Employee_ID'], task['Task_ID']))

print(f"Generated {len(valid_pairs)} valid assignment combinations.\n")
# print(valid_pairs)


Generated 233 valid assignment combinations.



In [103]:
# 4. Initialize OR-Tools Solver

solver = pywraplp.Solver.CreateSolver('GLOP')

# Decision Variables
x = {}
for (i, j) in valid_pairs:
    x[(i, j)] = solver.NumVar(0, solver.infinity(), f'x_{i}_{j}')

print(x)

# Objective Function
cost_dict = employees_df.set_index('Employee_ID')['Hourly_Cost'].to_dict()
objective = solver.Objective()
for (i, j) in valid_pairs:
    objective.SetCoefficient(x[(i, j)], float(cost_dict[i]))
objective.SetMinimization()

# Constraint 1: Demand
demand_dict = tasks_df.set_index('Task_ID')['Hours_Needed'].to_dict()
for j in tasks_df['Task_ID']:
    capable_employees = [i for (i, t_id) in valid_pairs if t_id == j]
    constraint = solver.Constraint(float(demand_dict[j]), float(demand_dict[j]), f"Demand_{j}")
    for i in capable_employees:
        constraint.SetCoefficient(x[(i, j)], 1)

# Constraint 2: Capacity
cap_dict = employees_df.set_index('Employee_ID')['Max_Hours'].to_dict()
for i in employees_df['Employee_ID']:
    assigned_tasks = [j for (e_id, j) in valid_pairs if e_id == i]
    constraint = solver.Constraint(0, float(cap_dict[i]), f"Capacity_{i}")
    for j in assigned_tasks:
        constraint.SetCoefficient(x[(i, j)], 1)

# 3. Generate Valid Pairs using Set Theory
valid_pairs = []
for _, emp in employees_df.iterrows():
    emp_skill_set = set(emp['Specific_Skills']) 
    for _, task in tasks_df.iterrows():
        task_req_set = set(task['Skills_Needed'])
        # subset check: employee must have ALL skills required by the task
        if task_req_set.issubset(emp_skill_set):
            valid_pairs.append((emp['Employee_ID'], task['Task_ID']))

print(f"Generated {len(valid_pairs)} valid assignment combinations.\n")

# 4. Initialize OR-Tools Solver

solver = pywraplp.Solver.CreateSolver('GLOP')

# Decision Variables
x = {}
for (i, j) in valid_pairs:
    x[(i, j)] = solver.NumVar(0, solver.infinity(), f'x_{i}_{j}')

print(x)

# Objective Function
cost_dict = employees_df.set_index('Employee_ID')['Hourly_Cost'].to_dict()
objective = solver.Objective()
for (i, j) in valid_pairs:
    objective.SetCoefficient(x[(i, j)], float(cost_dict[i]))
objective.SetMinimization()

# Constraint 1: Demand
demand_dict = tasks_df.set_index('Task_ID')['Hours_Needed'].to_dict()
for j in tasks_df['Task_ID']:
    capable_employees = [i for (i, t_id) in valid_pairs if t_id == j]
    constraint = solver.Constraint(float(demand_dict[j]), float(demand_dict[j]), f"Demand_{j}")
    for i in capable_employees:
        constraint.SetCoefficient(x[(i, j)], 1)

# Constraint 2: Capacity
cap_dict = employees_df.set_index('Employee_ID')['Max_Hours'].to_dict()
for i in employees_df['Employee_ID']:
    assigned_tasks = [j for (e_id, j) in valid_pairs if e_id == i]
    constraint = solver.Constraint(0, float(cap_dict[i]), f"Capacity_{i}")
    for j in assigned_tasks:
        constraint.SetCoefficient(x[(i, j)], 1)

# 5. Solve
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print(f"Status: OPTIMAL")
    print(f"Total Portfolio Cost: ${solver.Objective().Value():,.2f}")
    
    # Extract results
    results = []
    for (i, j) in valid_pairs:
        assigned_hours = x[(i, j)].solution_value()
        if assigned_hours > 0: 
            results.append({
                'Task_ID': j,
                'Employee_ID': i,
                'Assigned_Hours': assigned_hours,
                'Task_Cost': assigned_hours * cost_dict[i]
            })
    
    results_df = pd.DataFrame(results).sort_values(by=['Task_ID', 'Employee_ID']).reset_index(drop=True)
    
    print("\n--- Final Assignment Schedule ---")
    # display(results_df)
    
    # Analytics for Presentation: Did we use the contractor?
    contractor_hours = results_df[results_df['Employee_ID'] == 'EXT_CONTRACTOR']['Assigned_Hours'].sum()
    if contractor_hours > 0:
        print(f"\n🚨 BUSINESS ALERT: Internal workforce lacked capacity/skills.")
        print(f"🚨 Forced to outsource {contractor_hours:,.0f} hours to the External Contractor.")
        display(results_df[results_df['Employee_ID'] == 'EXT_CONTRACTOR'])
    else:
        print("\n✅ SUCCESS: All projects staffed purely by internal workforce.")

else:
    print("Solver failed. Status:", status)


{('E001', 'P009_T2'): x_E001_P009_T2, ('E002', 'P002_T2'): x_E002_P002_T2, ('E002', 'P004_T4'): x_E002_P004_T4, ('E002', 'P005_T4'): x_E002_P005_T4, ('E002', 'P010_T3'): x_E002_P010_T3, ('E002', 'P012_T1'): x_E002_P012_T1, ('E002', 'P013_T3'): x_E002_P013_T3, ('E003', 'P003_T2'): x_E003_P003_T2, ('E004', 'P009_T1'): x_E004_P009_T1, ('E004', 'P010_T1'): x_E004_P010_T1, ('E004', 'P010_T2'): x_E004_P010_T2, ('E004', 'P011_T2'): x_E004_P011_T2, ('E004', 'P012_T2'): x_E004_P012_T2, ('E004', 'P013_T2'): x_E004_P013_T2, ('E006', 'P003_T1'): x_E006_P003_T1, ('E006', 'P004_T2'): x_E006_P004_T2, ('E006', 'P007_T1'): x_E006_P007_T1, ('E006', 'P007_T2'): x_E006_P007_T2, ('E006', 'P007_T3'): x_E006_P007_T3, ('E006', 'P008_T2'): x_E006_P008_T2, ('E007', 'P004_T2'): x_E007_P004_T2, ('E007', 'P008_T2'): x_E007_P008_T2, ('E008', 'P005_T2'): x_E008_P005_T2, ('E008', 'P010_T1'): x_E008_P010_T1, ('E008', 'P011_T2'): x_E008_P011_T2, ('E008', 'P012_T2'): x_E008_P012_T2, ('E010', 'P004_T4'): x_E010_P004_T4, 